# 02 — Food Name Matcher: TF-IDF + Cosine Similarity

**Model:** TF-IDF character n-gram vectorizer + cosine similarity
- Toleran terhadap typo (char ngram 2-4)
- Alias mapping untuk variasi nama makanan Indonesia
- Pre-computed matrix untuk inference cepat (<5ms)

**Input:** `food_master_v3.parquet`

**Output:**
- `output/models/food_tfidf_vectorizer.pkl`
- `output/models/food_name_matrix.npy`
- `output/preprocessed/food_processed.parquet`
- `output/preprocessed/alias_map.json`
- `output/preprocessed/metadata.json`

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import os
import unicodedata
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs('output/models', exist_ok=True)
os.makedirs('output/preprocessed', exist_ok=True)

FOOD_MASTER_PATH = '../Model_Perencana_Makan/output/preprocessed/food_master_v3.parquet'
df = pd.read_parquet(FOOD_MASTER_PATH).reset_index(drop=True)
print(f'Loaded: {df.shape}')
df.head(3)

Loaded: (1346, 20)


,id,slug,name,category,cuisine,calories_per_portion,protein_g,fat_g,carbs_g,estimated_price_idr,is_halal,is_vegetarian,is_vegan,is_gluten_free,image_url,base_portion,base_portion_gram,fiber_g,is_active,cal_per_1000_idr
0,1,abon,Abon,STAPLE,INDONESIAN_GENERAL,280.0,9.2,28.4,0.0,12000,True,True,True,True,https://img-cdn.medkomtek.com/PbrY9X3ignQ8sVuj...,1 porsi,100,0.0,True,23.3
1,2,abon-haruwan,Abon haruwan,STAPLE,INDONESIAN_GENERAL,513.0,23.7,37.0,21.3,18500,True,True,True,True,https://img-global.cpcdn.com/recipes/cbf330fbd...,1 porsi,100,0.0,True,27.7
2,3,agar-agar,Agar-agar,STAPLE,INDONESIAN_GENERAL,1.0,0.0,0.2,0.0,8500,True,True,True,True,https://res.cloudinary.com/dk0z4ums3/image/upl...,1 porsi,100,0.0,False,0.1


In [2]:
# ── 1. Text Normalization ──
def normalize_food_name(text):
    """Lowercase, hapus accent, strip special chars untuk fuzzy matching."""
    text = str(text).lower().strip()
    # Hapus accent marks (é -> e, ñ -> n, dst)
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    # Hapus karakter non-alphanumeric kecuali spasi
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['name_normalized'] = df['name'].apply(normalize_food_name)
df['slug_normalized']  = df['slug'].apply(normalize_food_name)

print('Normalization sample:')
print(df[['name', 'name_normalized']].head(10).to_string())

Normalization sample:
                 name     name_normalized
0                Abon                abon
1        Abon haruwan        abon haruwan
2           Agar-agar           agar agar
3  Akar tonjong segar  akar tonjong segar
4       Aletoge segar       aletoge segar
5       Alpukat segar       alpukat segar
6  Ampas kacang hijau  ampas kacang hijau
7          Ampas Tahu          ampas tahu
8    Ampas tahu kukus    ampas tahu kukus
9   Ampas tahu mentah   ampas tahu mentah


In [3]:
# ── 2. Alias Dictionary (variasi nama makanan Indonesia) ──
ALIAS_MAP = {
    # canonical_name: [list of aliases/typos/variants]
    'nasi goreng':    ['nasgor', 'nasi goring', 'nasi gorng', 'nasigoreng'],
    'mie goreng':     ['mi goreng', 'mie goring', 'indomie goreng', 'mie gorng'],
    'ayam goreng':    ['ayam goring', 'ayam krispy', 'chicken goreng', 'ayam gorng'],
    'tempe':          ['tempeh', 'tahu tempe', 'tempe goreng'],
    'tahu':           ['tofu', 'tau fu', 'tauhu'],
    'kangkung':       ['water spinach', 'kankung', 'kangkung rebus', 'kangkung tumis'],
    'bayam':          ['spinach', 'bayem', 'bayam rebus', 'bayam tumis'],
    'telur':          ['egg', 'telor', 'telur goreng', 'telur rebus', 'telor ceplok'],
    'soto':           ['soto ayam', 'soto daging', 'soto babat', 'soto mie'],
    'rendang':        ['rendang daging', 'rendang sapi', 'beef rendang'],
    'gado gado':      ['gado-gado', 'gadogado', 'gado2'],
    'bakso':          ['baso', 'bakso sapi', 'bakso ayam', 'meatball soup'],
    'sate':           ['satay', 'sate ayam', 'sate kambing', 'sate maranggi'],
    'nasi putih':     ['white rice', 'steamed rice', 'plain rice', 'nasi'],
    'nasi merah':     ['red rice', 'brown rice', 'beras merah'],
    'singkong':       ['cassava', 'ubi kayu', 'ketela pohon'],
    'ubi jalar':      ['sweet potato', 'ubi', 'ketela rambat'],
    'kacang tanah':   ['peanut', 'kacang', 'groundnut'],
    'susu':           ['milk', 'susu sapi', 'dairy milk'],
    'es teh':         ['iced tea', 'teh es', 'es teh manis'],
    'air putih':      ['water', 'mineral water', 'air mineral'],
    'pisang':         ['banana', 'pisang goreng', 'banana fritter'],
    'apel':           ['apple', 'buah apel'],
    'jeruk':          ['orange', 'buah jeruk', 'jeruk manis'],
    'semangka':       ['watermelon', 'buah semangka'],
    'daging sapi':    ['beef', 'sapi', 'daging lembu'],
    'daging ayam':    ['chicken', 'ayam', 'chicken breast'],
    'ikan salmon':    ['salmon', 'ikan salm'],
    'ikan tuna':      ['tuna', 'canned tuna'],
    'udang':          ['shrimp', 'prawn', 'udang goreng'],
}

# Flatten alias map (alias -> canonical)
alias_map_flat = {}
for canonical, aliases in ALIAS_MAP.items():
    norm_canonical = normalize_food_name(canonical)
    for alias in aliases:
        alias_map_flat[normalize_food_name(alias)] = norm_canonical

with open('output/preprocessed/alias_map.json', 'w', encoding='utf-8') as f:
    json.dump(alias_map_flat, f, indent=2, ensure_ascii=False)
print(f'Alias map: {len(alias_map_flat)} entries saved.')

Alias map: 97 entries saved.


In [4]:
# ── 3. Build TF-IDF Vectorizer ──
# Gabungkan name + slug + category + cuisine sebagai corpus
corpus_df = df[['name_normalized', 'slug_normalized', 'category', 'cuisine']].copy()
corpus_df['combined'] = (
    corpus_df['name_normalized'] + ' ' +
    corpus_df['slug_normalized'] + ' ' +
    corpus_df['category'].str.lower() + ' ' +
    corpus_df['cuisine'].str.lower()
)

# Pakai char n-gram (2,4) → typo tolerant
vectorizer = TfidfVectorizer(
    analyzer='char_wb',    # character n-grams with word boundaries
    ngram_range=(2, 4),    # bigram sampai 4-gram
    max_features=8000,
    sublinear_tf=True,     # log normalization untuk term frequency
    min_df=1,
)

all_texts = corpus_df['combined'].tolist()
vectorizer.fit(all_texts)

# Pre-compute food matrix (1,346 x vocab_size)
food_matrix = vectorizer.transform(corpus_df['combined'])

print(f'TF-IDF fitted. Shape: {food_matrix.shape}')
print(f'Vocab size: {len(vectorizer.vocabulary_)}')
print(f'Sample top terms: {list(vectorizer.vocabulary_.keys())[:20]}')

TF-IDF fitted. Shape: (1346, 5141)
Vocab size: 5141
Sample top terms: [' a', 'ab', 'bo', 'on', 'n ', ' ab', 'abo', 'bon', 'on ', ' abo', 'abon', 'bon ', ' s', 'st', 'ta', 'ap', 'pl', 'le', 'e ', ' st']


In [5]:
# ── 4. Matching Function ──
def find_food(query, top_k=3, threshold=0.20):
    """
    Match query string ke food_master via TF-IDF cosine similarity.

    Args:
        query: nama makanan (dari Gemini Vision atau user input)
        top_k: jumlah kandidat terbaik yang dikembalikan
        threshold: minimum confidence score (0.0 - 1.0)

    Returns:
        list of dict dengan food match + confidence
    """
    # Normalize & apply alias mapping
    norm_query = normalize_food_name(query)
    if norm_query in alias_map_flat:
        norm_query = alias_map_flat[norm_query]

    # Vectorize & compute similarities
    q_vec = vectorizer.transform([norm_query])
    sims  = cosine_similarity(q_vec, food_matrix).flatten()

    # Get top-K
    top_indices = np.argsort(sims)[::-1][:top_k]
    results = []
    for idx in top_indices:
        sim = float(sims[idx])
        if sim < threshold:
            break
        row = df.iloc[idx]
        results.append({
            'food_id':    int(row['id']),
            'name':       row['name'],
            'category':   row['category'],
            'calories':   float(row['calories_per_portion']),
            'protein_g':  float(row['protein_g']),
            'fat_g':      float(row['fat_g']),
            'carbs_g':    float(row['carbs_g']),
            'is_halal':   bool(row['is_halal']),
            'confidence': round(sim, 4),
        })
    return results

# Test cepat
print(find_food('nasi goreng'))

[{'food_id': 924, 'name': 'Nasi Goreng ', 'category': 'STAPLE', 'calories': 276.0, 'protein_g': 3.2, 'fat_g': 3.2, 'carbs_g': 30.2, 'is_halal': True, 'confidence': 0.9273}, {'food_id': 921, 'name': 'Nasi', 'category': 'STAPLE', 'calories': 180.0, 'protein_g': 3.0, 'fat_g': 0.3, 'carbs_g': 39.8, 'is_halal': True, 'confidence': 0.6879}, {'food_id': 926, 'name': 'Nasi jagung', 'category': 'STAPLE', 'calories': 357.0, 'protein_g': 8.8, 'fat_g': 0.5, 'carbs_g': 79.5, 'is_halal': True, 'confidence': 0.5138}]


In [6]:
# ── 5. Evaluate Matcher ──
TEST_QUERIES = [
    # (query, expected_name_contains)
    ('nasi goreng',    'goreng'),      # exact
    ('nasgor',         'goreng'),      # singkatan alias
    ('telur ceplok',   'ceplok'),      # exact multi-word
    ('telor goreng',   'telur'),       # typo: telor->telur
    ('ayam gorng',     'ayam'),        # typo partial
    ('kangkung rebus', 'kangkung'),    # multi-word
    ('bakso sapi',     'bakso'),       # multi-word
    ('tempe goreng',   'tempe'),       # alias
    ('pizza',          None),          # OOD (tidak ada di DB)
    ('sushi',          None),          # OOD
    ('burger',         None),          # OOD
]

print('=== FOOD MATCHER EVALUATION ===')
correct, ood_correct = 0, 0
valid_queries  = [(q, e) for q, e in TEST_QUERIES if e is not None]
ood_queries    = [(q, e) for q, e in TEST_QUERIES if e is None]

for query, expected in TEST_QUERIES:
    results = find_food(query, top_k=1, threshold=0.20)
    if results:
        best = results[0]
        if expected is None:
            # OOD: should have low confidence
            ood_ok = best['confidence'] < 0.40
            status = 'OOD_OK' if ood_ok else 'OOD_MISS'
            if ood_ok: ood_correct += 1
        else:
            is_correct = expected.lower() in best['name'].lower()
            status = 'OK' if is_correct else 'MISS'
            if is_correct: correct += 1
        print(f'[{status:8s}] "{query:20s}" -> "{best["name"]}" (conf: {best["confidence"]:.3f})')
    else:
        status = 'NO_MATCH'
        if expected is None: ood_correct += 1
        print(f'[{status:8s}] "{query:20s}" -> tidak ditemukan')

print(f'\nMatching accuracy: {correct}/{len(valid_queries)} ({correct/len(valid_queries)*100:.1f}%)')
print(f'OOD detection:     {ood_correct}/{len(ood_queries)} ({ood_correct/len(ood_queries)*100:.1f}%)')

=== FOOD MATCHER EVALUATION ===
[OK      ] "nasi goreng         " -> "Nasi Goreng " (conf: 0.927)
[OK      ] "nasgor              " -> "Nasi Goreng " (conf: 0.927)
[OK      ] "telur ceplok        " -> "Telur Ayam ceplok" (conf: 0.878)
[MISS    ] "telor goreng        " -> "Daun Kelor" (conf: 0.474)
[OK      ] "ayam gorng          " -> "Ayam usus goreng" (conf: 0.726)
[OK      ] "kangkung rebus      " -> "Kangkung" (conf: 0.896)
[OK      ] "bakso sapi          " -> "Bakso " (conf: 0.943)
[OK      ] "tempe goreng        " -> "Tempe Goreng " (conf: 0.701)
[NO_MATCH] "pizza               " -> tidak ditemukan
[OOD_OK  ] "sushi               " -> "Susu Sapi" (conf: 0.364)
[OOD_MISS] "burger              " -> "Beef burger" (conf: 0.693)

Matching accuracy: 7/8 (87.5%)
OOD detection:     2/3 (66.7%)


In [7]:
# ── 6. Save Artifacts ──
# TF-IDF vectorizer
joblib.dump(vectorizer, 'output/models/food_tfidf_vectorizer.pkl', compress=3)
print(f'Vectorizer: {os.path.getsize("output/models/food_tfidf_vectorizer.pkl")/1024:.1f} KB')

# Pre-computed food matrix (dense, untuk cepat inference)
np.save('output/models/food_name_matrix.npy', food_matrix.toarray())
print(f'Food matrix: {food_matrix.toarray().nbytes/1024:.1f} KB')

# Food index (subset kolom penting)
food_index = df[[
    'id', 'name', 'category', 'cuisine',
    'calories_per_portion', 'protein_g', 'fat_g', 'carbs_g',
    'is_halal', 'is_vegetarian', 'is_vegan', 'estimated_price_idr'
]].copy()
food_index.to_parquet('output/preprocessed/food_processed.parquet', index=False)
print(f'Food processed: {len(food_index)} rows')

# Metadata
metadata = {
    'vectorizer_type':     'TF-IDF char_wb n-gram (2,4)',
    'n_foods':             int(len(df)),
    'vocab_size':          int(len(vectorizer.vocabulary_)),
    'alias_entries':       int(len(alias_map_flat)),
    'match_threshold':     0.20,
    'ood_threshold':       0.40,
    'top_k_default':       3,
    'inference_note':      'food_name_matrix.npy pre-computed, inference <5ms',
}
with open('output/preprocessed/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('\nOK All artifacts saved:')
for fname in sorted(os.listdir('output/models')):
    kb = os.path.getsize(f'output/models/{fname}') / 1024
    print(f'  output/models/{fname:40s} ({kb:.1f} KB)')

Vectorizer: 37.5 KB
Food matrix: 54060.8 KB
Food processed: 1346 rows

OK All artifacts saved:
  output/models/food_name_matrix.npy                     (54061.0 KB)
  output/models/food_tfidf_vectorizer.pkl                (37.5 KB)
